# AyurLLM --- QLoRA fine-tuning on KaggleSame structure as your MedQA notebook: install, authenticate, write the trainerout with `%%writefile`, smoke-run it, then run it properly and look at thedashboards. The trainer and its data layer are the parts that changed.### Read this before you run anything**Your Hugging Face token was hardcoded in the original notebook.** Anyone withthe notebook had your account. Revoke it at<https://huggingface.co/settings/tokens> and use Kaggle Secrets instead --- theauth cell below does that.### What this notebook trains onIt accepts **either** schema and detects which it has, so the same trainer worksfor both:| | Looks like | Task ||---|---|---|| **QA splits** | `question`, `option_a..e`, `correct_answer` | pick the option letter || **Raw treatise** | `sanskrit_text`, `transliteration`, `commentary` | translate / transliterate / explain |The raw corpus is worth training on directly. A verse and its commentary teachthe model the text itself; an MCQ only teaches it to pick a letter. Each chunkyields up to three tasks, and only when the field that would be the answer isactually present --- a chunk with empty commentary produces no translation taskrather than one with an empty target.On Charaka Samhita: **1,497 chunks -> 3,496 training pairs** (1,450 translate,1,396 explain, 650 transliterate), 33 chunks skipped as too thin.

## 1 --- Install

In [ ]:
!pip install -q -U \    transformers \    trl \    peft \    bitsandbytes \    datasets \    accelerate \    matplotlib

## 2 --- AuthenticateAdd your token under **Add-ons -> Secrets** as `HF_TOKEN`, then run this. Neverpaste it into a cell: notebooks get shared, committed and published, and thetoken goes with them.Only needed for gated models. Skip it entirely for open ones.

In [ ]:
import ostry:    from kaggle_secrets import UserSecretsClient    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")    from huggingface_hub import login    login(token=os.environ["HF_TOKEN"])    print("Hugging Face authentication complete (token from Kaggle Secrets).")except Exception as e:    print("No HF_TOKEN secret found (%s)." % type(e).__name__)    print("Fine for open models; add the secret if yours is gated.")

## 3 --- The data layerWritten to a file rather than defined inline so it can be imported by thetrainer and tested on its own. Everything here is schema handling: optionparsing, multi-answer keys, language routing, and the raw-chunk task builders.

In [ ]:
%%writefile ayur_data.py#!/usr/bin/env python3# -*- coding: utf-8 -*-"""Data layer for AyurLLM fine-tuning --- schema, prompts, formatting, auditing.=============================================================================Kept separate from the trainer so it can be imported and tested without a GPU,a model download, or CUDA. ``test_ayur_data.py`` exercises everything hereagainst the real split files.THE SCHEMA----------One JSON array per split, 41 fields per row. The ones that matter for training:    question         the stem    option_a..e      option text; e is empty except on multiple-selection items    correct_answer   "A", or "A,C" for multiple-selection    answer_text      the correct option's text    question_type    one of six formats    language         "en" | "hi"    subject_domain   used only for reportingWHY THE PROMPT LOOKS LIKE THIS------------------------------*Instruction matches the item's language.* A Hindi question with an Englishinstruction teaches the model that answer language is unrelated to questionlanguage, which is exactly wrong for a bilingual benchmark.*The instruction states how many answers to give.* Multiple-selection items wanttwo or three letters; every other format wants one. Without that, the modellearns to hedge and emits several letters on single-answer items.*Completion is the letter, optionally followed by the answer text.* BhashaBenchis scored on the letter, so letter-only keeps training and evaluation in the sameunits. ``--completion_style letter_text`` adds the text for models that benefitfrom generating the content, at the cost of a longer completion span."""from __future__ import annotationsimport jsonimport reimport unicodedatafrom collections import Counterfrom typing import Any, Dict, List, OptionalLETTERS = ["a", "b", "c", "d", "e"]DEVA = re.compile(r"[ऀ-ॿ]")# The six formats, with the instruction that tells the model what shape of answer# is wanted. Two languages each, because an English instruction on a Hindi item# is a training signal in the wrong direction.INSTRUCTION = {    "en": {        "Multiple selection": "Select ALL correct options. Answer with the option "                              "letters separated by commas, for example: A,C",        "Match the column": "Choose the option with the correct pairing. Answer "                            "with one option letter only.",        "Assertion or Reasoning": "Evaluate the Assertion and the Reason. Answer "                                  "with one option letter only.",        "Fill in the blanks": "Choose the term that fills [X]. Answer with one "                              "option letter only.",        "Assorted": "Answer with one option letter only.",        "MCQ": "Answer with one option letter only.",    },    "hi": {        "Multiple selection": "सभी सही विकल्प चुनें। उत्तर केवल विकल्प अक्षरों में दें, "                              "अल्पविराम से अलग करके, जैसे: A,C",        "Match the column": "सही जोड़ी वाला विकल्प चुनें। उत्तर केवल एक विकल्प अक्षर में दें।",        "Assertion or Reasoning": "अभिकथन और कारण का मूल्यांकन करें। उत्तर केवल एक "                                  "विकल्प अक्षर में दें।",        "Fill in the blanks": "[X] के स्थान पर सही शब्द चुनें। उत्तर केवल एक विकल्प अक्षर में दें।",        "Assorted": "उत्तर केवल एक विकल्प अक्षर में दें।",        "MCQ": "उत्तर केवल एक विकल्प अक्षर में दें।",    },}SYSTEM = {    "en": "You are an expert in Ayurveda, trained on the classical Sanskrit "          "treatises. Answer examination questions accurately and concisely.",    "hi": "आप आयुर्वेद के विशेषज्ञ हैं, जो शास्त्रीय संस्कृत ग्रंथों में प्रशिक्षित हैं। "          "परीक्षा प्रश्नों का सटीक और संक्षिप्त उत्तर दें।",}def clean(s: Any) -> str:    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", str(s or ""))).strip()def load_split(path: str) -> List[Dict[str, Any]]:    """Accepts a JSON array or JSON Lines; both appear across this project."""    with open(path, encoding="utf-8") as fh:        head = fh.read(2048)        fh.seek(0)        if head.lstrip().startswith("["):            return json.load(fh)        return [json.loads(l) for l in fh if l.strip()]def options_of(row: Dict[str, Any]) -> Dict[str, str]:    """Present, non-empty options only. Rows carry five slots; most use four."""    return {L: clean(row.get("option_%s" % L))            for L in LETTERS if clean(row.get("option_%s" % L))}def answer_keys(row: Dict[str, Any], opts: Dict[str, str]) -> List[str]:    """Upper-case letters that exist as options. Handles "A" and "A,C" alike."""    raw = str(row.get("correct_answer", ""))    keys = [k.strip().upper() for k in raw.split(",") if k.strip()]    return [k for k in keys if k.lower() in opts]def infer_language(row: Dict[str, Any]) -> str:    """Normalise the label; fall back to the script of the stem when absent."""    raw = str(row.get("language", "")).strip().lower()    if raw in ("hi", "hin", "hindi", "हिन्दी", "हिंदी"):        return "hi"    if raw in ("en", "eng", "english"):        return "en"    q = clean(row.get("question"))    return "hi" if len(DEVA.findall(q)) / max(len(q), 1) > 0.30 else "en"# Some rows are open-ended by construction --- "explain this verse", "name the# sthana and chapter" --- so they have no options at all. They are ~2.5% of the# corpus and every one carries an answer_text, which makes them the most# content-rich rows available: answering them requires generating Ayurvedic# explanation rather than picking a letter. Dropping them for lack of options# would throw away exactly the material that teaches domain knowledge, so they# are trained as free-text generation instead.OPEN_INSTRUCTION = {    "en": "Answer concisely, based on the classical text.",    "hi": "शास्त्रीय ग्रंथ के आधार पर संक्षेप में उत्तर दें।",}def is_open_ended(row: Dict[str, Any]) -> bool:    """No usable options, but a usable answer --- a free-text item."""    opts = options_of(row)    if len(opts) >= 2 and answer_keys(row, opts):        return False    return bool(clean(row.get("answer_text")))def build_user_prompt(row: Dict[str, Any], allow_open: bool = True) -> Optional[str]:    opts = options_of(row)    keys = answer_keys(row, opts)    lang = infer_language(row)    if len(opts) < 2 or not keys:        if allow_open and is_open_ended(row):            return "%s\n\n%s" % (clean(row.get("question")), OPEN_INSTRUCTION[lang])        return None    qtype = row.get("question_type", "MCQ")    instr = INSTRUCTION[lang].get(qtype, INSTRUCTION[lang]["MCQ"])    option_block = "\n".join("%s. %s" % (L.upper(), opts[L])                             for L in LETTERS if L in opts)    return "%s\n\n%s\n\n%s" % (clean(row.get("question")), option_block, instr)def format_example(row: Dict[str, Any], tokenizer,                   completion_style: str = "letter",                   use_system: bool = True,                   allow_open: bool = True,                   open_answer_chars: int = 600) -> Dict[str, str]:    """Row -> {"prompt", "completion"} for TRL's native completion-only loss.    Returns empty strings when the row cannot be rendered; the caller drops those    rather than training on a malformed item.    """    user = build_user_prompt(row, allow_open=allow_open)    if user is None:        return {"prompt": "", "completion": ""}    opts = options_of(row)    keys = answer_keys(row, opts)    lang = infer_language(row)    if not keys:        # Free-text item. Capping the answer keeps a handful of very long verse        # expositions from dominating the loss and blowing past max_seq_length.        answer = clean(row.get("answer_text"))[:open_answer_chars]    else:        answer = ",".join(keys)        if completion_style == "letter_text":            answer = "%s. %s" % (answer, " ; ".join(opts[k.lower()] for k in keys))    eos = getattr(tokenizer, "eos_token", "") or ""    template = getattr(tokenizer, "chat_template", None)    if template:        msgs = ([{"role": "system", "content": SYSTEM[lang]}] if use_system else [])        msgs = msgs + [{"role": "user", "content": user}]        prompt_only = tokenizer.apply_chat_template(            msgs, tokenize=False, add_generation_prompt=True)        full = tokenizer.apply_chat_template(            msgs + [{"role": "assistant", "content": answer}], tokenize=False)        if full.startswith(prompt_only) and len(full) > len(prompt_only):            return {"prompt": prompt_only, "completion": full[len(prompt_only):]}        # Some templates do not prefix-match cleanly. Falling back keeps the run        # alive; the masking check downstream still confirms a non-empty span.        return {"prompt": prompt_only, "completion": answer + eos}    head = SYSTEM[lang] + "\n\n" if use_system else ""    return {"prompt": "%sUser: %s\nAnswer:" % (head, user),            "completion": " %s%s" % (answer, eos)}# --------------------------------------------------------------------------# Auditing --- run before training, not after it fails# --------------------------------------------------------------------------def audit_rows(rows: List[Dict[str, Any]]) -> Dict[str, Any]:    """Composition, plus the two defects that silently break a run."""    return {        "rows": len(rows),        "unusable_missing_options": sum(1 for r in rows if len(options_of(r)) < 2),        "unusable_missing_answer_key": sum(            1 for r in rows if not answer_keys(r, options_of(r))),        "by_type": dict(Counter(r.get("question_type") for r in rows).most_common()),        "by_language": dict(Counter(infer_language(r) for r in rows)),        "by_book": dict(Counter(r.get("source_book") for r in rows).most_common()),        "multi_answer_rows": sum(1 for r in rows                                 if len(answer_keys(r, options_of(r))) > 1),        "open_ended_rows": sum(1 for r in rows if is_open_ended(r)),        "answer_position": dict(Counter(            k for r in rows for k in answer_keys(r, options_of(r)))),    }def token_length_audit(pairs: List[Dict[str, str]], tokenizer,                       max_seq_length: int, sample: int = 4000) -> Dict[str, Any]:    """Percentiles of prompt+completion length, and how much would be truncated.    Devanagari costs several tokens per character on most tokenizers, so a    max_seq_length chosen for an English corpus can silently cut the answer off a    large share of the Hindi half --- and a truncated completion means the loss is    computed over nothing.    """    import random    rnd = random.Random(0)    take = pairs if len(pairs) <= sample else rnd.sample(pairs, sample)    lens, comp_lens = [], []    for p in take:        pl = len(tokenizer(p["prompt"], add_special_tokens=False)["input_ids"])        cl = len(tokenizer(p["completion"], add_special_tokens=False)["input_ids"])        lens.append(pl + cl)        comp_lens.append(cl)    lens.sort()    def pct(q):        return lens[min(int(q * len(lens)), len(lens) - 1)] if lens else 0    over = sum(1 for x in lens if x > max_seq_length)    return {        "sampled": len(lens),        "total_tokens_p50": pct(0.50), "total_tokens_p90": pct(0.90),        "total_tokens_p99": pct(0.99), "total_tokens_max": lens[-1] if lens else 0,        "completion_tokens_mean": round(sum(comp_lens) / max(len(comp_lens), 1), 2),        "completion_tokens_min": min(comp_lens) if comp_lens else 0,        "rows_over_max_seq_length": over,        "pct_over_max_seq_length": round(100 * over / max(len(lens), 1), 2),        "max_seq_length": max_seq_length,        "all_token_lengths": lens,          # for the histogram plot    }def prepare(rows: List[Dict[str, Any]], tokenizer, completion_style="letter",            use_system=True, allow_open=True, open_answer_chars=600):    """Format and drop unusable rows. Returns (pairs, dropped, n_open)."""    out, dropped, n_open = [], 0, 0    for r in rows:        p = format_example(r, tokenizer, completion_style, use_system,                           allow_open, open_answer_chars)        if not p["prompt"] or not p["completion"].strip():            dropped += 1            continue        if allow_open and is_open_ended(r):            n_open += 1        out.append(p)    return out, dropped, n_open# ==========================================================================# Raw treatise chunks --- the other schema this corpus ships in# ==========================================================================# The QA splits above are one input. The other is the source corpus itself:# one row per verse, with sanskrit_text, transliteration and commentary. Those# rows have no question and no options, so nothing above applies to them --- but# they are the most direct way to teach the model the texts, because the target# is the actual translation rather than a letter.## Three tasks are derived per chunk, and only when the field that would be the# answer is actually present:##   translate      Sanskrit verse -> commentary / English meaning#   transliterate  Devanagari -> IAST#   explain        cited topic -> what the text says about it## A chunk with an empty commentary yields no translate task rather than a task# with an empty target, which would train the model to emit nothing.CHUNK_KEYS = {"sanskrit_text", "commentary", "chunk_id"}CHUNK_TASKS = {    "translate": {        "en": "Translate this verse from {cite} into English.",        "hi": "{cite} के इस श्लोक का हिन्दी में अनुवाद करें।",    },    "transliterate": {        "en": "Transliterate this verse from {cite} into IAST.",        "hi": "{cite} के इस श्लोक का IAST में लिप्यंतरण करें।",    },    "explain": {        "en": "According to {cite}, explain: {topic}",        "hi": "{cite} के अनुसार समझाइए: {topic}",    },}def is_chunk_row(row: Dict[str, Any]) -> bool:    """True when the row is a raw treatise chunk rather than a QA item."""    return "question" not in row and CHUNK_KEYS.issubset(set(row))def detect_schema(rows: List[Dict[str, Any]]) -> str:    """'qa' | 'chunk' | 'mixed', from the first few hundred rows."""    sample = rows[:500]    if not sample:        return "qa"    chunk = sum(1 for r in sample if is_chunk_row(r))    if chunk == 0:        return "qa"    if chunk == len(sample):        return "chunk"    return "mixed"def chunk_citation(row: Dict[str, Any]) -> str:    bits = [clean(row.get("book") or row.get("source_book") or "the classical text")]    if clean(row.get("sthana")):        bits.append(clean(row.get("sthana")))    if row.get("chapter_number") not in (None, ""):        bits.append("Chapter %s" % row.get("chapter_number"))    if clean(row.get("verse_id")):        bits.append("verse %s" % clean(row.get("verse_id")))    return ", ".join(bits)def chunk_examples(row: Dict[str, Any], min_target: int = 40,                   max_target_chars: int = 1200) -> List[Dict[str, str]]:    """Every usable (instruction, input, target) triple this chunk supports."""    out = []    cite = chunk_citation(row)    sanskrit = clean(row.get("sanskrit_text"))    translit = clean(row.get("transliteration"))    comm = clean(row.get("commentary"))    topic = clean(row.get("topic_heading")) or clean(row.get("chapter_title"))    lang = "hi" if len(DEVA.findall(comm)) / max(len(comm), 1) > 0.30 else "en"    if sanskrit and len(comm) >= min_target:        out.append({"instruction": CHUNK_TASKS["translate"][lang].format(cite=cite),                    "input": sanskrit, "target": comm[:max_target_chars],                    "task": "translate", "language": lang})    # Only when the transliteration is really IAST. Some rows carry English prose    # in that field --- training on those teaches the model that transliteration    # means translation.    if sanskrit and translit and not DEVA.search(translit)             and len(translit) >= min_target             and len(re.findall(r"[āīūṛṝḷṅñṭḍṇśṣḥṁṃ]", translit)) >= 2:        out.append({"instruction": CHUNK_TASKS["transliterate"][lang].format(cite=cite),                    "input": sanskrit, "target": translit[:max_target_chars],                    "task": "transliterate", "language": lang})    if topic and len(comm) >= min_target and topic.lower() not in comm.lower()[:60]:        out.append({"instruction": CHUNK_TASKS["explain"][lang].format(                        cite=cite, topic=topic),                    "input": "", "target": comm[:max_target_chars],                    "task": "explain", "language": lang})    return outdef format_chunk_example(ex: Dict[str, str], tokenizer,                         use_system: bool = True) -> Dict[str, str]:    lang = ex.get("language", "en")    user = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")    answer = ex["target"]    eos = getattr(tokenizer, "eos_token", "") or ""    template = getattr(tokenizer, "chat_template", None)    if template:        msgs = ([{"role": "system", "content": SYSTEM[lang]}] if use_system else [])        msgs = msgs + [{"role": "user", "content": user}]        prompt_only = tokenizer.apply_chat_template(            msgs, tokenize=False, add_generation_prompt=True)        full = tokenizer.apply_chat_template(            msgs + [{"role": "assistant", "content": answer}], tokenize=False)        if full.startswith(prompt_only) and len(full) > len(prompt_only):            return {"prompt": prompt_only, "completion": full[len(prompt_only):]}        return {"prompt": prompt_only, "completion": answer + eos}    head = SYSTEM[lang] + "\n\n" if use_system else ""    return {"prompt": "%sUser: %s\nAssistant:" % (head, user),            "completion": " %s%s" % (answer, eos)}def prepare_chunks(rows: List[Dict[str, Any]], tokenizer, use_system=True,                   max_target_chars=1200):    """Raw chunks -> prompt/completion pairs. Returns (pairs, skipped, by_task)."""    out, skipped = [], 0    by_task = Counter()    for r in rows:        exs = chunk_examples(r, max_target_chars=max_target_chars)        if not exs:            skipped += 1            continue        for e in exs:            p = format_chunk_example(e, tokenizer, use_system)            if p["prompt"] and p["completion"].strip():                out.append(p)                by_task[e["task"]] += 1    return out, skipped, dict(by_task)def prepare_any(rows: List[Dict[str, Any]], tokenizer, **kw):    """Route by detected schema. Returns (pairs, info dict)."""    schema = detect_schema(rows)    if schema == "chunk":        pairs, skipped, by_task = prepare_chunks(            rows, tokenizer, kw.get("use_system", True),            kw.get("max_target_chars", 1200))        return pairs, {"schema": "chunk", "input_rows": len(rows),                       "pairs": len(pairs), "chunks_skipped": skipped,                       "by_task": by_task}    if schema == "qa":        pairs, dropped, n_open = prepare(            rows, tokenizer, kw.get("completion_style", "letter"),            kw.get("use_system", True), kw.get("allow_open", True),            kw.get("open_answer_chars", 600))        return pairs, {"schema": "qa", "input_rows": len(rows),                       "pairs": len(pairs), "dropped": dropped,                       "open_ended": n_open}    qa_rows = [r for r in rows if not is_chunk_row(r)]    ch_rows = [r for r in rows if is_chunk_row(r)]    a, ia = prepare_any(qa_rows, tokenizer, **kw)    b, ib = prepare_any(ch_rows, tokenizer, **kw)    return a + b, {"schema": "mixed", "qa": ia, "chunk": ib,                   "pairs": len(a) + len(b)}

## 4 --- The trainerYour script, with the data layer swapped and throughput work added:- **`group_by_length=True`** --- the single biggest win here. Stems run from 40 to  900 tokens, so length-sorted batching removes most of the padding.- **TF32** matmuls on Ampere+, real dataloader workers, pinned memory.- **Pre-flight length audit.** Devanagari costs several tokens per character, so  a `max_seq_length` picked for English silently truncates the answer off a large  share of the corpus --- and a truncated completion means loss over nothing. The  script warns, and by default drops those rows rather than training on broken  ones.- **Six dashboard panels** instead of four: loss, LR, throughput, gradient norm,  GPU spikes, and a token-length histogram with the truncation line drawn on it.- **Answer-position audit.** If the correct answer sits at one letter far more  often than chance, a model can score well by always picking it.

In [ ]:
%%writefile finetune_ayur.py"""QLoRA / SFT Fine-Tuning for AyurLLM===================================Adapted from the MedQA trainer. Same architecture --- typed config, native TRLcompletion-only loss, hardware autotuning, telemetry, dashboards --- retargeted atthe Ayurvedic corpus and tuned for throughput.WHAT CHANGED, AND WHY---------------------1.  DATA LAYER REPLACED. MedQA ships one flat schema from the Hub. This corpus is    three local JSON files with `option_a..e`, `correct_answer` that may hold    several letters, six question formats and two languages. All of that lives in    `ayur_data.py` so it can be tested without a GPU --- run `test_ayur_data.py`    before you burn an hour of compute on a schema bug.2.  REAL VALIDATION SPLIT. The MedQA script carved eval out of train with    `train_test_split`. Here `validation.json` already exists, built at fact    granularity so no fact straddles the two. Re-splitting train would silently    undo that, so it is not done.3.  PRE-FLIGHT AUDITS. Composition, answer-position balance and token-length    percentiles are computed and saved before training starts. The length audit    matters more than it sounds: Devanagari costs several tokens per character,    so a `max_seq_length` chosen for English can truncate the answer off a large    share of the Hindi half --- and a truncated completion means loss over nothing.    The script warns loudly if that is about to happen.4.  THROUGHPUT WORK.      - `group_by_length=True`  batches similar lengths together, which on a        corpus whose stems run from 40 to 600 tokens removes most of the padding.        This is the single biggest win available here.      - TF32 matmuls enabled on Ampere+.      - `dataloader_num_workers` / `pin_memory` / `persistent_workers` set from a        sensible default rather than left at 0.      - Tokenization mapped with `num_proc` and cached.      - Optional `--auto_batch_size` probes VRAM and picks a per-device batch        size instead of making you guess.5.  MORE DASHBOARDS. The original four (loss, LR, batch size, GPU) plus    throughput (samples/sec), grad-norm, and a token-length histogram with the    truncation line drawn on it.Usage    python finetune_ayur.py \        --model_name_or_path Qwen/Qwen2.5-3B-Instruct \        --data_dir ../ayur_pipeline/final \        --output_dir ./ayur_qlora_out \        --num_train_epochs 2 \        --per_device_train_batch_size 8 \        --gradient_accumulation_steps 4 \        --learning_rate 2e-4 \        --max_seq_length 768    # prove the data layer works first, no GPU needed:    python test_ayur_data.py --data_dir ../ayur_pipeline/final"""import sysimport os# Reduce CUDA allocator fragmentation across repeated in-kernel runs (e.g. via# Jupyter %run) --- must be set before torch is imported / initializes CUDA.os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")import gcimport jsonimport timeimport loggingimport threadingfrom dataclasses import dataclass, fieldfrom typing import Optional, Dict, Any, Listimport torchfrom datasets import Datasetfrom peft import LoraConfig, prepare_model_for_kbit_trainingfrom transformers import (    AutoModelForCausalLM,    AutoTokenizer,    BitsAndBytesConfig,    HfArgumentParser,    TrainerCallback,    PreTrainedTokenizerBase,)from transformers.trainer_utils import get_last_checkpointfrom trl import SFTConfig, SFTTrainerimport matplotlibmatplotlib.use("Agg")  # headless-safe backend for servers/containersimport matplotlib.pyplot as pltsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))import ayur_data as ADlogging.basicConfig(    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",    datefmt="%Y-%m-%d %H:%M:%S",    level=logging.INFO,    handlers=[logging.StreamHandler(sys.stdout)],)logger = logging.getLogger("AyurLLM-Trainer")# ============================================================================# Dataclass Arguments# ============================================================================@dataclassclass ModelArguments:    model_name_or_path: str = field(        metadata={"help": "Pretrained model id or local path."}    )    use_4bit: bool = field(default=True, metadata={"help": "4-bit NF4 QLoRA."})    lora_r: int = field(default=32, metadata={        "help": "LoRA rank. 32 rather than MedQA's 16: this corpus asks the model "                "to absorb domain vocabulary in two scripts, which needs more "                "adapter capacity than picking an option letter on English text."})    lora_alpha: int = field(default=64, metadata={"help": "LoRA scaling (2x rank)."})    lora_dropout: float = field(default=0.05, metadata={"help": "LoRA dropout."})    lora_target_modules: str = field(        default="q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj",        metadata={"help": "Comma-separated module names to adapt."})@dataclassclass DataArguments:    data_dir: str = field(        default="../ayur_pipeline/final",        metadata={"help": "Folder holding train.json / validation.json / test.json."})    train_file: str = field(default="train.json", metadata={        "help": "QA split, or a raw treatise .jsonl (sanskrit_text/commentary). "                "The schema is detected, not configured."})    eval_file: str = field(default="validation.json", metadata={        "help": "Held-out file. Leave missing on raw-chunk data and --eval_frac "                "carves one out by chunk_id."})    eval_frac: float = field(default=0.02, metadata={        "help": "Only used when eval_file is absent. Split is by chunk_id, so no "                "verse appears in both halves."})    max_target_chars: int = field(default=1200, metadata={        "help": "Cap on a chunk-task target, so one long commentary cannot "                "dominate the loss."})    max_seq_length: int = field(default=768, metadata={        "help": "Token cap. Check the length audit the script prints --- Devanagari "                "items are much longer in tokens than they look in characters."})    max_train_samples: Optional[int] = field(default=None)    max_eval_samples: Optional[int] = field(default=None)    completion_style: str = field(default="letter", metadata={        "help": "'letter' (A / A,C) or 'letter_text' (letter plus the option text)."})    use_system_prompt: bool = field(default=True)    include_open_ended: bool = field(default=True, metadata={        "help": "Train the ~2.5%% of rows that have no options (verse expositions, "                "source lookups) as free-text generation instead of dropping them. "                "They carry the richest domain content in the corpus."})    open_answer_chars: int = field(default=600, metadata={        "help": "Cap on free-text answer length, so a few long expositions do not "                "dominate the loss or blow past max_seq_length."})    filter_question_types: Optional[str] = field(default=None, metadata={        "help": "Comma-separated question types to keep. Default keeps all six."})    filter_language: Optional[str] = field(default=None, metadata={        "help": "'en' or 'hi' to train on one language only. Default keeps both."})    drop_over_length: bool = field(default=True, metadata={        "help": "Drop rows longer than max_seq_length instead of truncating them. "                "Truncation removes the completion, which makes the row a silent "                "no-op that still costs compute."})@dataclassclass PerfArguments:    auto_batch_size: bool = field(default=False, metadata={        "help": "Probe VRAM and choose per_device_train_batch_size."})    tokenize_num_proc: int = field(default=4)    dataloader_workers: int = field(default=-1, metadata={        "help": "-1 picks min(8, cpu_count)."})    enable_tf32: bool = field(default=True)    group_by_length: bool = field(default=True, metadata={        "help": "Batch similar-length sequences together. Biggest single "                "throughput win on this corpus; disable only to debug."})@dataclassclass OptimizerSwitchArguments:    optimizer_switch_step: Optional[int] = field(default=None)    secondary_optimizer: str = field(default="adamw_torch")    secondary_learning_rate: Optional[float] = field(default=None)    secondary_weight_decay: Optional[float] = field(default=None)# ============================================================================# Utilities & Engineering Assertions# ============================================================================def verify_masking_integrity(sample: Dict[str, Any], tokenizer) -> None:    """Confirms a non-empty completion span exists for loss computation.    Modern TRL masks the prompt natively when the dataset has separate 'prompt'    and 'completion' columns and SFTConfig(completion_only_loss=True) is set. This    tokenizes prompt vs prompt+completion independently to confirm the trainer has    something to compute loss over.    """    logger.info("Running pre-flight Label Masking Integrity Check...")    prompt_ids = tokenizer(sample["prompt"], add_special_tokens=False)["input_ids"]    full_ids = tokenizer(sample["prompt"] + sample["completion"],                         add_special_tokens=False)["input_ids"]    prompt_len = len(prompt_ids)    completion_len = len(full_ids) - prompt_len    logger.info("Token Analysis -> Prompt: %d | Completion (loss target): %d",                prompt_len, completion_len)    if completion_len <= 0:        raise RuntimeError(            "CRITICAL: completion span is 0 tokens. The model would train on "            "nothing. Check ayur_data.format_example() and the chat template.")    logger.info("Masking verification passed.")def log_gpu_footprint(tag: str = "Status"):    if torch.cuda.is_available():        logger.info("[%s] VRAM Allocated: %.2fGB | Reserved: %.2fGB | Peak: %.2fGB",                    tag, torch.cuda.memory_allocated() / 1024 ** 3,                    torch.cuda.memory_reserved() / 1024 ** 3,                    torch.cuda.max_memory_allocated() / 1024 ** 3)def suggest_batch_size(max_seq_length: int, use_4bit: bool) -> int:    """Rough per-device batch size from free VRAM, sequence length and precision.    Deliberately conservative --- an OOM three hours into a run costs far more than    a slightly smaller batch, and gradient accumulation recovers the effective    size anyway.    """    if not torch.cuda.is_available():        return 4    free = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3    budget = free * (0.55 if use_4bit else 0.35)    per_sample_gb = (max_seq_length / 1024.0) * (0.35 if use_4bit else 0.9)    bs = max(1, int(budget / max(per_sample_gb, 0.05)))    for cap in (64, 32, 16, 8, 4, 2, 1):        if bs >= cap:            return cap    return 1# ============================================================================# Telemetry# ============================================================================class GPUSpikeMonitor:    """Background sampler recording VRAM at fixed intervals, so short-lived    spikes (attention peak, optimizer step) are not missed --- the Trainer's own    logging cadence is far too coarse to catch them.    """    def __init__(self, interval_sec: float = 1.0):        self.interval_sec = interval_sec        self._stop_event = threading.Event()        self._thread: Optional[threading.Thread] = None        self.timestamps: List[float] = []        self.allocated_gb: List[float] = []        self.reserved_gb: List[float] = []        self.utilization: List[float] = []        self._start_time = None    def _sample_loop(self):        while not self._stop_event.is_set():            if torch.cuda.is_available():                self.timestamps.append(time.time() - self._start_time)                self.allocated_gb.append(torch.cuda.memory_allocated() / 1024 ** 3)                self.reserved_gb.append(torch.cuda.memory_reserved() / 1024 ** 3)                try:                    self.utilization.append(float(torch.cuda.utilization()))                except Exception:                    self.utilization.append(float("nan"))            self._stop_event.wait(self.interval_sec)    def start(self):        if not torch.cuda.is_available():            logger.warning("CUDA unavailable --- GPU monitor collects nothing.")            return        self._start_time = time.time()        self._thread = threading.Thread(target=self._sample_loop, daemon=True)        self._thread.start()        logger.info("GPU spike monitor started (every %.1fs).", self.interval_sec)    def stop(self):        if self._thread is not None:            self._stop_event.set()            self._thread.join(timeout=5)            logger.info("GPU monitor stopped. %d samples.", len(self.timestamps))    def peak_allocated_gb(self) -> float:        return max(self.allocated_gb) if self.allocated_gb else 0.0    def peak_reserved_gb(self) -> float:        return max(self.reserved_gb) if self.reserved_gb else 0.0class MetricsLoggerCallback(TrainerCallback):    """Writes step metrics to JSONL and buffers them for plotting."""    def __init__(self, filepath: str):        self.filepath = filepath        self.history: List[Dict[str, Any]] = []    def on_log(self, args, state, control, logs=None, **kwargs):        if logs:            entry = dict(logs)            entry["step"] = state.global_step            entry["epoch"] = state.epoch            entry["wall_time"] = time.time()            with open(self.filepath, "a", encoding="utf-8") as f:                f.write(json.dumps(entry) + "\n")            self.history.append(entry)class CheckpointLossReporter(TrainerCallback):    """Persists the latest train/eval loss beside every checkpoint."""    def __init__(self):        self.last_train_loss: Optional[float] = None        self.last_eval_loss: Optional[float] = None    def on_log(self, args, state, control, logs=None, **kwargs):        if not logs:            return        if "loss" in logs:            self.last_train_loss = logs["loss"]        if "eval_loss" in logs:            self.last_eval_loss = logs["eval_loss"]    def on_save(self, args, state, control, **kwargs):        ckpt_dir = os.path.join(args.output_dir, "checkpoint-%d" % state.global_step)        logger.info("Checkpoint saved -> step %d | train_loss=%s | eval_loss=%s",                    state.global_step, self.last_train_loss, self.last_eval_loss)        try:            os.makedirs(ckpt_dir, exist_ok=True)            with open(os.path.join(ckpt_dir, "checkpoint_loss_summary.json"), "w") as f:                json.dump({"global_step": state.global_step, "epoch": state.epoch,                           "last_train_loss": self.last_train_loss,                           "last_eval_loss": self.last_eval_loss}, f, indent=2)        except FileNotFoundError:            logger.warning("Checkpoint dir not ready; JSONL trace still has it.")def build_optimizer(name: str, model_params, lr: float, weight_decay: float = 0.0):    name = name.lower()    params = [p for p in model_params if p.requires_grad]    if name == "adamw_torch":        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)    if name == "sgd":        return torch.optim.SGD(params, lr=lr, weight_decay=weight_decay, momentum=0.9)    if name == "rmsprop":        return torch.optim.RMSprop(params, lr=lr, weight_decay=weight_decay)    if name == "adafactor":        from transformers.optimization import Adafactor        return Adafactor(params, lr=lr, weight_decay=weight_decay,                         scale_parameter=False, relative_step=False)    if name in ("adamw_8bit", "paged_adamw_8bit"):        import bitsandbytes as bnb        cls = bnb.optim.PagedAdamW8bit if name == "paged_adamw_8bit" else bnb.optim.AdamW8bit        return cls(params, lr=lr, weight_decay=weight_decay)    raise ValueError("Unknown secondary_optimizer '%s'." % name)class OptimizerSwitchCallback(TrainerCallback):    """Swaps the optimizer mid-run at a given global step."""    def __init__(self, switch_step, optimizer_name, lr, weight_decay):        self.switch_step = switch_step        self.optimizer_name = optimizer_name        self.lr_override = lr        self.weight_decay_override = weight_decay        self.trainer = None          # wired up after Trainer construction        self._switched = False        self.switch_history: List[Dict[str, Any]] = []    def on_step_end(self, args, state, control, **kwargs):        if not self.switch_step or self._switched or state.global_step < self.switch_step:            return        if self.trainer is None:            logger.warning("OptimizerSwitchCallback fired without .trainer set.")            self._switched = True            return        lr = self.lr_override if self.lr_override is not None else args.learning_rate        wd = (self.weight_decay_override if self.weight_decay_override is not None              else args.weight_decay)        logger.info("Switching optimizer at step %d -> '%s' (lr=%s, wd=%s)",                    state.global_step, self.optimizer_name, lr, wd)        try:            self.trainer.optimizer = build_optimizer(                self.optimizer_name, self.trainer.model.parameters(), lr, wd)            self.switch_history.append({"step": state.global_step,                                        "optimizer": self.optimizer_name, "lr": lr})            logger.info("Optimizer switch complete.")        except Exception as e:            logger.error("Optimizer switch failed, continuing with previous: %s", e)        finally:            self._switched = True# ============================================================================# Dashboards# ============================================================================def build_dashboard_figure(metrics_history, gpu_monitor, batch_size,                           grad_accum_steps, switch_history=None,                           length_audit=None):    """One 3x2 figure: loss, LR, throughput, grad-norm, GPU, token lengths.    Shared by the live in-notebook callback and the final PNG export so the two    cannot drift apart.    """    steps = [m["step"] for m in metrics_history if "loss" in m]    train_losses = [m["loss"] for m in metrics_history if "loss" in m]    eval_steps = [m["step"] for m in metrics_history if "eval_loss" in m]    eval_losses = [m["eval_loss"] for m in metrics_history if "eval_loss" in m]    lr_steps = [m["step"] for m in metrics_history if "learning_rate" in m]    lrs = [m["learning_rate"] for m in metrics_history if "learning_rate" in m]    gn_steps = [m["step"] for m in metrics_history if "grad_norm" in m]    gns = [m["grad_norm"] for m in metrics_history if "grad_norm" in m]    tp_steps = [m["step"] for m in metrics_history if "train_samples_per_second" in m]    tps = [m["train_samples_per_second"] for m in metrics_history           if "train_samples_per_second" in m]    fig, axes = plt.subplots(3, 2, figsize=(13, 12))    ax_loss, ax_lr = axes[0, 0], axes[0, 1]    ax_tp, ax_gn = axes[1, 0], axes[1, 1]    ax_gpu, ax_len = axes[2, 0], axes[2, 1]    if train_losses:        ax_loss.plot(steps, train_losses, label="Train", color="tab:blue", lw=1.2)    if eval_losses:        ax_loss.plot(eval_steps, eval_losses, label="Eval", color="tab:orange",                     marker="o", ms=3)    for sw in (switch_history or []):        ax_loss.axvline(sw["step"], color="tab:red", ls=":", alpha=0.8)    ax_loss.set_title("Training / Eval Loss")    ax_loss.set_xlabel("Global Step"); ax_loss.set_ylabel("Loss")    ax_loss.grid(alpha=0.3)    if train_losses or eval_losses:        ax_loss.legend()    if lrs:        ax_lr.plot(lr_steps, lrs, color="tab:green")    ax_lr.set_title("Learning Rate Schedule")    ax_lr.set_xlabel("Global Step"); ax_lr.set_ylabel("LR")    ax_lr.grid(alpha=0.3)    if tps:        ax_tp.plot(tp_steps, tps, color="tab:cyan")    else:        eff = batch_size * grad_accum_steps        ax_tp.plot(steps, [eff] * len(steps), color="tab:red", ls="--",                   label="Effective batch")        ax_tp.plot(steps, [batch_size] * len(steps), color="tab:purple",                   label="Per-device batch")        if steps:            ax_tp.legend()    ax_tp.set_title("Throughput (samples/s)" if tps else "Batch Size")    ax_tp.set_xlabel("Global Step")    ax_tp.grid(alpha=0.3)    if gns:        ax_gn.plot(gn_steps, gns, color="tab:brown", lw=1.0)    ax_gn.set_title("Gradient Norm")    ax_gn.set_xlabel("Global Step"); ax_gn.set_ylabel("||g||")    ax_gn.grid(alpha=0.3)    if gpu_monitor.timestamps:        ax_gpu.plot(gpu_monitor.timestamps, gpu_monitor.allocated_gb,                    label="Allocated", color="tab:blue", lw=1.0)        ax_gpu.plot(gpu_monitor.timestamps, gpu_monitor.reserved_gb,                    label="Reserved", color="tab:gray", alpha=0.6, lw=1.0)        peak = gpu_monitor.peak_allocated_gb()        ax_gpu.axhline(peak, color="tab:red", ls="--",                       label="Peak %.2fGB" % peak)        ax_gpu.legend(fontsize=8)    ax_gpu.set_title("GPU Memory / Spikes")    ax_gpu.set_xlabel("Wall-clock (s)"); ax_gpu.set_ylabel("VRAM (GB)")    ax_gpu.grid(alpha=0.3)    if length_audit and length_audit.get("all_token_lengths"):        L = length_audit["all_token_lengths"]        ax_len.hist(L, bins=50, color="tab:olive", alpha=0.85)        ax_len.axvline(length_audit["max_seq_length"], color="tab:red", ls="--",                       label="max_seq_length=%d" % length_audit["max_seq_length"])        ax_len.axvline(length_audit["total_tokens_p99"], color="tab:orange", ls=":",                       label="p99=%d" % length_audit["total_tokens_p99"])        ax_len.legend(fontsize=8)    ax_len.set_title("Token Length Distribution")    ax_len.set_xlabel("Tokens (prompt + completion)"); ax_len.set_ylabel("Rows")    ax_len.grid(alpha=0.3)    fig.tight_layout()    return figclass LiveDashboardCallback(TrainerCallback):    """Redraws the dashboard in-place in a notebook cell. No-op outside IPython."""    def __init__(self, metrics_callback, gpu_monitor, batch_size, grad_accum_steps,                 refresh_every_n_logs=1, optimizer_switch_callback=None,                 length_audit=None):        self.metrics_callback = metrics_callback        self.gpu_monitor = gpu_monitor        self.batch_size = batch_size        self.grad_accum_steps = grad_accum_steps        self.refresh_every_n_logs = max(1, refresh_every_n_logs)        self.optimizer_switch_callback = optimizer_switch_callback        self.length_audit = length_audit        self._log_count = 0        self._display = self._clear = self._Image = None        try:            from IPython.display import display, clear_output, Image            self._display, self._clear, self._Image = display, clear_output, Image        except ImportError:            logger.info("IPython absent --- live dashboard off; PNGs still saved.")    def on_log(self, args, state, control, logs=None, **kwargs):        if self._display is None or not logs:            return        self._log_count += 1        if self._log_count % self.refresh_every_n_logs != 0:            return        try:            import io            sw = (self.optimizer_switch_callback.switch_history                  if self.optimizer_switch_callback else None)            fig = build_dashboard_figure(                self.metrics_callback.history, self.gpu_monitor, self.batch_size,                self.grad_accum_steps, switch_history=sw,                length_audit=self.length_audit)            # Render to PNG bytes explicitly: matplotlib.use("Agg") strips            # Jupyter's inline formatter, so display(fig) would print a repr.            buf = io.BytesIO()            fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")            plt.close(fig)            buf.seek(0)            self._clear(wait=True)            self._display(self._Image(data=buf.getvalue()))        except Exception as e:            logger.warning("Live dashboard redraw failed (non-fatal): %s", e)def plot_training_dashboards(metrics_history, gpu_monitor, batch_size,                             grad_accum_steps, output_dir, switch_history=None,                             length_audit=None):    """Writes the combined dashboard plus individual PNGs."""    plots_dir = os.path.join(output_dir, "training_plots")    os.makedirs(plots_dir, exist_ok=True)    fig = build_dashboard_figure(metrics_history, gpu_monitor, batch_size,                                 grad_accum_steps, switch_history, length_audit)    fig.savefig(os.path.join(plots_dir, "dashboard.png"), dpi=150,                bbox_inches="tight")    plt.close(fig)    steps = [m["step"] for m in metrics_history if "loss" in m]    train_losses = [m["loss"] for m in metrics_history if "loss" in m]    eval_steps = [m["step"] for m in metrics_history if "eval_loss" in m]    eval_losses = [m["eval_loss"] for m in metrics_history if "eval_loss" in m]    if train_losses or eval_losses:        plt.figure(figsize=(8, 5))        if train_losses:            plt.plot(steps, train_losses, label="Train Loss", color="tab:blue")        if eval_losses:            plt.plot(eval_steps, eval_losses, label="Eval Loss",                     color="tab:orange", marker="o")        for sw in (switch_history or []):            plt.axvline(sw["step"], color="tab:red", ls=":", alpha=0.8)        plt.xlabel("Global Step"); plt.ylabel("Loss")        plt.title("Training / Eval Loss"); plt.legend(); plt.grid(alpha=0.3)        plt.tight_layout()        plt.savefig(os.path.join(plots_dir, "loss_curve.png"), dpi=150)        plt.close()    if gpu_monitor.timestamps:        plt.figure(figsize=(8, 5))        plt.plot(gpu_monitor.timestamps, gpu_monitor.allocated_gb,                 label="Allocated (GB)", color="tab:blue")        plt.plot(gpu_monitor.timestamps, gpu_monitor.reserved_gb,                 label="Reserved (GB)", color="tab:gray", alpha=0.6)        peak = gpu_monitor.peak_allocated_gb()        plt.axhline(peak, color="tab:red", ls="--", label="Peak: %.2fGB" % peak)        plt.xlabel("Wall-clock Time (s)"); plt.ylabel("VRAM (GB)")        plt.title("GPU Memory Usage / Spikes"); plt.legend(); plt.grid(alpha=0.3)        plt.tight_layout()        plt.savefig(os.path.join(plots_dir, "gpu_memory_spikes.png"), dpi=150)        plt.close()    logger.info("Training dashboards saved to: %s", plots_dir)# ============================================================================# Main# ============================================================================def main():    parser = HfArgumentParser((ModelArguments, DataArguments, PerfArguments,                              OptimizerSwitchArguments, SFTConfig))    model_args, data_args, perf_args, optim_switch_args, sft_config = \        parser.parse_args_into_dataclasses()    logger.info("Initializing AyurLLM fine-tuning for: %s",                model_args.model_name_or_path)    log_gpu_footprint("Pre-Flight Setup")    if perf_args.enable_tf32 and torch.cuda.is_available():        torch.backends.cuda.matmul.allow_tf32 = True        torch.backends.cudnn.allow_tf32 = True        logger.info("TF32 matmuls enabled.")    os.makedirs(sft_config.output_dir, exist_ok=True)    # ---- 1. Load the pre-built splits ---------------------------------    train_path = os.path.join(data_args.data_dir, data_args.train_file)    eval_path = os.path.join(data_args.data_dir, data_args.eval_file)    if not os.path.exists(train_path):        raise SystemExit("Missing training file: %s" % train_path)    train_rows = AD.load_split(train_path)    schema = AD.detect_schema(train_rows)    logger.info("Detected input schema: %s", schema)    if os.path.exists(eval_path):        eval_rows = AD.load_split(eval_path)    elif schema == "chunk":        # No held-out file, and chunk rows have no fact_id. Splitting on        # chunk_id keeps every task derived from one verse on the same side ---        # a random row split would put the translation of a verse in train and        # its transliteration in eval, which is the same leak by another name.        import random as _rnd        ids = sorted({str(r.get("chunk_id")) for r in train_rows})        _rnd.Random(sft_config.seed).shuffle(ids)        n_eval = max(1, int(len(ids) * data_args.eval_frac))        held = set(ids[:n_eval])        eval_rows = [r for r in train_rows if str(r.get("chunk_id")) in held]        train_rows = [r for r in train_rows if str(r.get("chunk_id")) not in held]        logger.info("Carved eval by chunk_id: %d chunks held out (%d rows)",                    len(held), len(eval_rows))    else:        raise SystemExit("Missing eval file: %s" % eval_path)    logger.info("Loaded -> train %d | validation %d", len(train_rows), len(eval_rows))    logger.info("Validation comes from validation.json, not a re-split of train: "                "the splits were built at fact granularity and re-splitting would "                "undo that guarantee.")    if data_args.filter_question_types:        keep = {t.strip() for t in data_args.filter_question_types.split(",")}        train_rows = [r for r in train_rows if r.get("question_type") in keep]        eval_rows = [r for r in eval_rows if r.get("question_type") in keep]        logger.info("Filtered to types %s -> train %d | val %d",                    sorted(keep), len(train_rows), len(eval_rows))    if data_args.filter_language:        lang = data_args.filter_language.strip().lower()        train_rows = [r for r in train_rows if AD.infer_language(r) == lang]        eval_rows = [r for r in eval_rows if AD.infer_language(r) == lang]        logger.info("Filtered to language '%s' -> train %d | val %d",                    lang, len(train_rows), len(eval_rows))    if data_args.max_train_samples:        train_rows = train_rows[:data_args.max_train_samples]    if data_args.max_eval_samples:        eval_rows = eval_rows[:data_args.max_eval_samples]    audit = ({"train": AD.audit_rows(train_rows),              "validation": AD.audit_rows(eval_rows)} if schema == "qa"             else {"note": "schema=%s; QA audits do not apply" % schema})    pos = {}    if schema == "qa":        logger.info("Train composition -> %s", audit["train"]["by_type"])        logger.info("Train languages   -> %s", audit["train"]["by_language"])        logger.info("Answer positions  -> %s", audit["train"]["answer_position"])        pos = audit["train"]["answer_position"]    if pos:        top = max(pos.values()) / max(sum(pos.values()), 1)        if top > 0.40:            logger.warning("Answer-position skew: one letter holds %.1f%% of "                           "answers. A model can score well by always picking it. "                           "Consider re-running the quality gate with "                           "--fix-position.", 100 * top)    # ---- 2. Tokenizer --------------------------------------------------    tokenizer = AutoTokenizer.from_pretrained(        model_args.model_name_or_path,        trust_remote_code=sft_config.trust_remote_code)    if tokenizer.pad_token is None:        tokenizer.pad_token = tokenizer.eos_token    tokenizer.padding_side = "right"    # ---- 3. Format to prompt/completion --------------------------------    kw = dict(completion_style=data_args.completion_style,              use_system=data_args.use_system_prompt,              allow_open=data_args.include_open_ended,              open_answer_chars=data_args.open_answer_chars,              max_target_chars=data_args.max_target_chars)    train_pairs, info_tr = AD.prepare_any(train_rows, tokenizer, **kw)    eval_pairs, info_ev = AD.prepare_any(eval_rows, tokenizer, **kw)    logger.info("Train formatting: %s", info_tr)    logger.info("Eval  formatting: %s", info_ev)    dropped_tr = info_tr.get("dropped", info_tr.get("chunks_skipped", 0))    dropped_ev = info_ev.get("dropped", info_ev.get("chunks_skipped", 0))    open_tr = info_tr.get("open_ended", 0)    open_ev = info_ev.get("open_ended", 0)    logger.info("Formatted -> train %d (dropped %d, open-ended %d) | "                "val %d (dropped %d, open-ended %d)",                len(train_pairs), dropped_tr, open_tr,                len(eval_pairs), dropped_ev, open_ev)    if data_args.include_open_ended and open_tr:        logger.info("%d open-ended rows are trained as free-text generation "                    "rather than discarded --- they carry explanation content that "                    "letter-picking items do not.", open_tr)    if not train_pairs:        raise SystemExit("No usable training rows after formatting.")    # ---- 4. Length audit, before anything expensive --------------------    length_audit = AD.token_length_audit(train_pairs, tokenizer,                                         data_args.max_seq_length)    logger.info("Token lengths -> p50 %d | p90 %d | p99 %d | max %d",                length_audit["total_tokens_p50"], length_audit["total_tokens_p90"],                length_audit["total_tokens_p99"], length_audit["total_tokens_max"])    if length_audit["pct_over_max_seq_length"] > 2.0:        logger.warning(            "%.1f%% of rows exceed max_seq_length=%d. Truncation removes the "            "completion, so those rows train on nothing while still costing "            "compute. Raise --max_seq_length to about %d, or keep "            "--drop_over_length so they are removed rather than silently broken.",            length_audit["pct_over_max_seq_length"], data_args.max_seq_length,            length_audit["total_tokens_p99"])    if data_args.drop_over_length:        def fits(p):            n = len(tokenizer(p["prompt"] + p["completion"],                              add_special_tokens=False)["input_ids"])            return n <= data_args.max_seq_length        before = len(train_pairs)        train_pairs = [p for p in train_pairs if fits(p)]        eval_pairs = [p for p in eval_pairs if fits(p)]        logger.info("Dropped %d over-length training rows (%d remain).",                    before - len(train_pairs), len(train_pairs))    train_ds = Dataset.from_list(train_pairs)    eval_ds = Dataset.from_list(eval_pairs)    verify_masking_integrity(train_ds[0], tokenizer)    # ---- 5. Hardware dispatch & quantization ---------------------------    attn_implementation = "sdpa"    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:        try:            import flash_attn  # noqa: F401            attn_implementation = "flash_attention_2"        except ImportError:            pass    logger.info("Attention backend: '%s'", attn_implementation)    quant_config = None    if model_args.use_4bit:        compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16        quant_config = BitsAndBytesConfig(            load_in_4bit=True, bnb_4bit_quant_type="nf4",            bnb_4bit_compute_dtype=compute_dtype, bnb_4bit_use_double_quant=True)    model = AutoModelForCausalLM.from_pretrained(        model_args.model_name_or_path,        quantization_config=quant_config,        device_map="auto",        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,        attn_implementation=attn_implementation,        trust_remote_code=sft_config.trust_remote_code)    if model_args.use_4bit:        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)    peft_config = LoraConfig(        r=model_args.lora_r, lora_alpha=model_args.lora_alpha,        lora_dropout=model_args.lora_dropout, bias="none", task_type="CAUSAL_LM",        target_modules=[m.strip() for m in model_args.lora_target_modules.split(",")])    # ---- 6. Trainer configuration --------------------------------------    jsonl_log_path = os.path.join(sft_config.output_dir, "training_metrics.jsonl")    sft_config.completion_only_loss = True    # TRL defaults to loss_type="chunked_nll", which patches lm_head.forward. On    # some architectures lm_head.forward is already a functools.partial and the    # patch crashes reading __func__. The unchunked path sidesteps it.    if not getattr(sft_config, "loss_type", None) or sft_config.loss_type == "chunked_nll":        sft_config.loss_type = "nll"    sft_config.max_seq_length = data_args.max_seq_length    sft_config.optim = getattr(sft_config, "optim", None) or "paged_adamw_8bit"    sft_config.gradient_checkpointing = True    sft_config.gradient_checkpointing_kwargs = {"use_reentrant": False}    sft_config.bf16 = torch.cuda.is_bf16_supported()    sft_config.fp16 = not torch.cuda.is_bf16_supported()    if perf_args.auto_batch_size:        bs = suggest_batch_size(data_args.max_seq_length, model_args.use_4bit)        logger.info("Auto batch size -> %d per device", bs)        sft_config.per_device_train_batch_size = bs        sft_config.per_device_eval_batch_size = bs    sft_config.group_by_length = perf_args.group_by_length    workers = (perf_args.dataloader_workers if perf_args.dataloader_workers >= 0               else min(8, os.cpu_count() or 2))    sft_config.dataloader_num_workers = workers    sft_config.dataloader_pin_memory = True    if workers > 0:        sft_config.dataloader_persistent_workers = True    if getattr(sft_config, "save_strategy", None) in (None, "no"):        sft_config.save_strategy = "steps"    if not getattr(sft_config, "save_steps", None):        sft_config.save_steps = 200    if not getattr(sft_config, "logging_steps", None):        sft_config.logging_steps = 10    if getattr(sft_config, "save_total_limit", None) is None:        sft_config.save_total_limit = 3    if getattr(sft_config, "eval_strategy", "no") in (None, "no"):        sft_config.eval_strategy = sft_config.save_strategy    sft_config.load_best_model_at_end = True    sft_config.metric_for_best_model = "eval_loss"    sft_config.greater_is_better = False    sft_config.report_to = getattr(sft_config, "report_to", None) or []    metrics_callback = MetricsLoggerCallback(jsonl_log_path)    checkpoint_reporter = CheckpointLossReporter()    gpu_monitor = GPUSpikeMonitor(interval_sec=1.0)    optimizer_switch_callback = OptimizerSwitchCallback(        optim_switch_args.optimizer_switch_step,        optim_switch_args.secondary_optimizer,        optim_switch_args.secondary_learning_rate,        optim_switch_args.secondary_weight_decay)    live_dashboard = LiveDashboardCallback(        metrics_callback, gpu_monitor,        sft_config.per_device_train_batch_size,        sft_config.gradient_accumulation_steps,        refresh_every_n_logs=1,        optimizer_switch_callback=optimizer_switch_callback,        length_audit=length_audit)    # ---- 7. Save the pre-flight report ---------------------------------    la = {k: v for k, v in length_audit.items() if k != "all_token_lengths"}    preflight = {        "model": model_args.model_name_or_path,        "data_dir": data_args.data_dir,        "schema": schema, "audit": audit, "length_audit": la,        "formatting": {"train": info_tr, "eval": info_ev},        "formatted": {"train": len(train_ds), "validation": len(eval_ds),                      "dropped_train": dropped_tr, "dropped_eval": dropped_ev,                      "open_ended_train": open_tr, "open_ended_eval": open_ev},        "effective_batch": (sft_config.per_device_train_batch_size                            * sft_config.gradient_accumulation_steps                            * max(1, getattr(sft_config, "world_size", 1) or 1)),        "hyperparameters": {            "learning_rate": sft_config.learning_rate,            "num_train_epochs": sft_config.num_train_epochs,            "per_device_train_batch_size": sft_config.per_device_train_batch_size,            "gradient_accumulation_steps": sft_config.gradient_accumulation_steps,            "max_seq_length": data_args.max_seq_length,            "lora_r": model_args.lora_r, "lora_alpha": model_args.lora_alpha,            "lora_dropout": model_args.lora_dropout,            "optim": sft_config.optim, "bf16": sft_config.bf16,            "group_by_length": sft_config.group_by_length,            "attn_implementation": attn_implementation,            "warmup_ratio": getattr(sft_config, "warmup_ratio", None),            "lr_scheduler_type": str(getattr(sft_config, "lr_scheduler_type", "")),            "weight_decay": sft_config.weight_decay,        },    }    with open(os.path.join(sft_config.output_dir, "preflight_report.json"),              "w", encoding="utf-8") as f:        json.dump(preflight, f, ensure_ascii=False, indent=2)    logger.info("Effective batch size: %d", preflight["effective_batch"])    trainer = SFTTrainer(        model=model, args=sft_config,        train_dataset=train_ds, eval_dataset=eval_ds,        peft_config=peft_config, processing_class=tokenizer,        callbacks=[metrics_callback, checkpoint_reporter, live_dashboard,                   optimizer_switch_callback])    optimizer_switch_callback.trainer = trainer    log_gpu_footprint("Post-Initialization")    # ---- 8. Auto-resume -------------------------------------------------    resume_checkpoint = sft_config.resume_from_checkpoint    if not resume_checkpoint:        detected = get_last_checkpoint(sft_config.output_dir)        if detected:            resume_checkpoint = detected            logger.info("Resuming from checkpoint: %s", resume_checkpoint)        else:            logger.info("No checkpoint found --- training from scratch.")    # ---- 9. Train --------------------------------------------------------    logger.info("Starting fine-tuning...")    gpu_monitor.start()    try:        trainer.train(resume_from_checkpoint=resume_checkpoint)    except torch.cuda.OutOfMemoryError:        logger.error(            "CUDA OOM. Lower --per_device_train_batch_size and raise "            "--gradient_accumulation_steps to keep the effective batch, lower "            "--max_seq_length (check the length audit above for a safe value), "            "or restart the kernel to clear fragmentation from a prior run.")        gc.collect()        if torch.cuda.is_available():            torch.cuda.empty_cache()        raise    finally:        gpu_monitor.stop()    # ---- 10. Save --------------------------------------------------------    logger.info("Saving adapters and tokenizer to: %s", sft_config.output_dir)    trainer.save_model(sft_config.output_dir)    tokenizer.save_pretrained(sft_config.output_dir)    with open(os.path.join(sft_config.output_dir, "run_config_audit.json"), "w") as f:        json.dump({"model_args": model_args.__dict__,                   "data_args": data_args.__dict__,                   "perf_args": perf_args.__dict__,                   "optimizer_switch_args": optim_switch_args.__dict__}, f, indent=2)    metrics = trainer.evaluate()    logger.info("Final evaluation: %s", metrics)    with open(os.path.join(sft_config.output_dir, "final_metrics.json"), "w") as f:        json.dump({"eval": metrics,                   "peak_vram_allocated_gb": gpu_monitor.peak_allocated_gb(),                   "peak_vram_reserved_gb": gpu_monitor.peak_reserved_gb()},                  f, indent=2)    plot_training_dashboards(        metrics_callback.history, gpu_monitor,        sft_config.per_device_train_batch_size,        sft_config.gradient_accumulation_steps,        sft_config.output_dir,        optimizer_switch_callback.switch_history, length_audit)    logger.info("Peak GPU allocation: %.2fGB", gpu_monitor.peak_allocated_gb())    # ---- 11. VRAM cleanup, for repeated %run in one kernel ---------------    logger.info("Releasing references and clearing CUDA cache...")    try:        del trainer.optimizer    except Exception:        pass    del trainer    del model    gc.collect()    if torch.cuda.is_available():        torch.cuda.empty_cache()        torch.cuda.reset_peak_memory_stats()    log_gpu_footprint("Post-Cleanup")    logger.info("Pipeline execution complete.")if __name__ == "__main__":    main()

## 5 --- Pre-flight check (no GPU needed, takes seconds)Run this before committing to a long job. It exercises every path the trainertakes through the data --- loading, formatting, chat templates, masking, thelength audit --- with a stub tokenizer.A schema bug found here costs seconds. The same bug found on the GPU costshowever long it takes to reach the first eval.

In [ ]:
%%writefile test_ayur_data.py#!/usr/bin/env python3# -*- coding: utf-8 -*-"""Pre-flight test for the AyurLLM fine-tuning data layer. No GPU, no model download.==================================================================================Run this before launching a training job. It exercises every code path thetrainer will take through the data --- loading, formatting, chat-templatehandling, masking, the length audit and the plotting --- against the real splitfiles, using a stub tokenizer that mimics a chat model closely enough to catchschema bugs.A schema bug found here costs seconds. The same bug found on a GPU costs howeverlong it takes the run to reach the first eval.    python test_ayur_data.py --data_dir ../ayur_pipeline/final    python test_ayur_data.py --data_dir ../ayur_pipeline/final --real-tokenizer Qwen/Qwen2.5-3B-InstructExit code is 1 on any failure, so it can gate a job."""from __future__ import annotationsimport argparseimport jsonimport osimport reimport syssys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))import ayur_data as ADclass StubTokenizer:    """Minimal stand-in for a chat tokenizer.    Mimics the two behaviours that matter: a chat template whose    add_generation_prompt output is a strict prefix of the full rendering, and a    tokenizer callable returning input_ids. Token counts are approximate --- one    token per ~3.5 Latin characters, ~1.6 per Devanagari character, which is    roughly how real BPE tokenizers treat these scripts and enough to make the    length audit meaningful.    """    eos_token = "<|im_end|>"    chat_template = "stub"    pad_token = None    def apply_chat_template(self, messages, tokenize=False,                            add_generation_prompt=False):        out = []        for m in messages:            out.append("<|im_start|>%s\n%s<|im_end|>\n" % (m["role"], m["content"]))        s = "".join(out)        if add_generation_prompt:            s += "<|im_start|>assistant\n"        return s    def __call__(self, text, add_special_tokens=False):        deva = len(re.findall(r"[ऀ-ॿ]", text))        other = len(text) - deva        n = int(deva / 1.6 + other / 3.5) + 1        return {"input_ids": list(range(n))}def check(cond, msg, failures):    if not cond:        failures.append(msg)        print("   FAIL  %s" % msg)    return conddef main():    ap = argparse.ArgumentParser()    ap.add_argument("--data_dir", required=True)    ap.add_argument("--max_seq_length", type=int, default=768)    ap.add_argument("--completion_style", default="letter",                    choices=["letter", "letter_text"])    ap.add_argument("--real-tokenizer", dest="real_tokenizer", default="",                    help="HF model id; uses the real tokenizer instead of the stub")    ap.add_argument("--plot", default="", help="write a token-length PNG here")    ap.add_argument("--no-open-ended", dest="no_open_ended", action="store_true",                    help="drop option-less rows instead of training them free-text")    a = ap.parse_args()    failures = []    if a.real_tokenizer:        from transformers import AutoTokenizer        tok = AutoTokenizer.from_pretrained(a.real_tokenizer, trust_remote_code=True)        if tok.pad_token is None:            tok.pad_token = tok.eos_token        print("Using real tokenizer: %s" % a.real_tokenizer)    else:        tok = StubTokenizer()        print("Using stub tokenizer (no network needed)")    # ---------------- 1. files load --------------------------------------    print("\n1. Loading splits")    splits = {}    for name in ("train", "validation", "test"):        p = os.path.join(a.data_dir, "%s.json" % name)        if not os.path.exists(p):            p = os.path.join(a.data_dir, "%s.jsonl" % name)        if not os.path.exists(p):            print("   skip %s (not found)" % name)            continue        rows = AD.load_split(p)        splits[name] = rows        print("   %-11s %6d rows  %2d fields" % (name, len(rows), len(rows[0])))    check(bool(splits.get("train")), "train split missing or empty", failures)    # ---------------- 1b. which schema? ----------------------------------    schema = AD.detect_schema(splits.get("train", []))    print("\n   detected schema: %s" % schema)    if schema == "chunk":        # Raw treatise rows have no options and no answer key, so every QA check        # below is meaningless on them. Run the chunk-specific checks instead.        print("\n2. Chunk-task audit")        ok = True        for name, rows in splits.items():            pairs, info = AD.prepare_any(rows, tok)            print("   %-11s %5d chunks -> %5d pairs | skipped %d | %s"                  % (name, len(rows), len(pairs), info.get("chunks_skipped", 0),                     info.get("by_task")))            check(len(pairs) > 0, "%s: no chunk tasks produced" % name, failures)            check(info.get("chunks_skipped", 0) < 0.5 * max(len(rows), 1),                  "%s: over half the chunks yielded nothing" % name, failures)            if name == "train":                empty = 0                for pr in pairs[:2000]:                    pl = len(tok(pr["prompt"], add_special_tokens=False)["input_ids"])                    fl = len(tok(pr["prompt"] + pr["completion"],                                 add_special_tokens=False)["input_ids"])                    if fl - pl <= 0:                        empty += 1                print("   masking: %d of %d with an empty completion span"                      % (empty, min(len(pairs), 2000)))                check(empty == 0, "%d chunk pairs have no completion" % empty, failures)                la = AD.token_length_audit(pairs, tok, a.max_seq_length)                print("   lengths: p50 %d | p90 %d | p99 %d | over cap %.2f%%"                      % (la["total_tokens_p50"], la["total_tokens_p90"],                         la["total_tokens_p99"], la["pct_over_max_seq_length"]))                for pr in pairs[:2]:                    print("   --- sample ---")                    print("   prompt tail: ...%s" % pr["prompt"][-150:].replace("\n", " | "))                    print("   completion : %s" % pr["completion"][:110].replace("\n", " "))        print("\n%s" % ("PASS --- chunk data layer is sound" if not failures                        else "FAIL --- %d problem(s):" % len(failures)))        for f in failures:            print("  - %s" % f)        sys.exit(0 if not failures else 1)    # ---------------- 2. schema audit ------------------------------------    print("\n2. Schema audit")    for name, rows in splits.items():        au = AD.audit_rows(rows)        print("   %-11s types=%d langs=%s multi_answer=%d"              % (name, len(au["by_type"]), au["by_language"], au["multi_answer_rows"]))        print("               unusable: no options=%d, no answer key=%d"              % (au["unusable_missing_options"], au["unusable_missing_answer_key"]))        bad = au["unusable_missing_options"] + au["unusable_missing_answer_key"]        check(bad < 0.20 * max(len(rows), 1),              "%s: %d/%d rows unusable (>20%%)" % (name, bad, len(rows)), failures)        pos = au["answer_position"]        if pos:            top = max(pos.values()) / max(sum(pos.values()), 1)            print("               answer position: %s  (max share %.1f%%)"                  % (pos, 100 * top))            if top > 0.40:                print("   WARN  one letter holds %.1f%% of answers --- a model can "                      "score well by always picking it" % (100 * top))    # ---------------- 3. formatting --------------------------------------    print("\n3. Formatting to prompt/completion")    pairs = {}    for name, rows in splits.items():        p, dropped, n_open = AD.prepare(rows, tok, a.completion_style,                                        allow_open=not a.no_open_ended)        pairs[name] = p        print("   %-11s %6d formatted, %d dropped, %d open-ended"              % (name, len(p), dropped, n_open))        check(len(p) > 0, "%s: nothing formatted" % name, failures)        check(dropped < 0.20 * max(len(rows), 1),              "%s: dropped %d/%d (>20%%)" % (name, dropped, len(rows)), failures)    # ---------------- 4. masking integrity -------------------------------    print("\n4. Masking integrity (completion span must be non-empty)")    tr = pairs.get("train", [])    empty = 0    for p in tr[:2000]:        pl = len(tok(p["prompt"], add_special_tokens=False)["input_ids"])        fl = len(tok(p["prompt"] + p["completion"],                     add_special_tokens=False)["input_ids"])        if fl - pl <= 0:            empty += 1    print("   checked %d rows, %d with an empty completion span" % (min(len(tr), 2000), empty))    check(empty == 0, "%d rows have a zero-length completion span" % empty, failures)    # ---------------- 5. answer correctness ------------------------------    print("\n5. Completion content")    bad_letter = 0    multi_seen = 0    for row in splits["train"][:5000]:        opts = AD.options_of(row)        keys = AD.answer_keys(row, opts)        if not keys:            continue        if len(keys) > 1:            multi_seen += 1        pc = AD.format_example(row, tok, a.completion_style)        if not pc["completion"]:            continue        # every key letter must appear in the completion        if not all(k in pc["completion"] for k in keys):            bad_letter += 1    print("   multi-answer rows seen: %d | completions missing a key letter: %d"          % (multi_seen, bad_letter))    check(bad_letter == 0, "%d completions omit a correct-answer letter" % bad_letter,          failures)    # ---------------- 6. language routing --------------------------------    print("\n6. Language routing")    mism = 0    for row in splits["train"][:5000]:        lang = AD.infer_language(row)        up = AD.build_user_prompt(row)        if up is None:            continue        # Open-ended rows use their own instruction table, so check that one.        if AD.is_open_ended(row):            instr = AD.OPEN_INSTRUCTION[lang]        else:            instr = AD.INSTRUCTION[lang].get(row.get("question_type"),                                             AD.INSTRUCTION[lang]["MCQ"])        if instr not in up:            mism += 1    print("   prompts whose instruction language does not match the item: %d" % mism)    check(mism == 0, "%d prompts have a mismatched instruction language" % mism,          failures)    # ---------------- 7. length audit ------------------------------------    print("\n7. Token length audit")    la = AD.token_length_audit(tr, tok, a.max_seq_length)    print("   p50 %d | p90 %d | p99 %d | max %d"          % (la["total_tokens_p50"], la["total_tokens_p90"],             la["total_tokens_p99"], la["total_tokens_max"]))    print("   completion tokens: mean %.2f, min %d"          % (la["completion_tokens_mean"], la["completion_tokens_min"]))    print("   over max_seq_length=%d: %d rows (%.2f%%)"          % (a.max_seq_length, la["rows_over_max_seq_length"],             la["pct_over_max_seq_length"]))    check(la["completion_tokens_min"] > 0,          "some completions tokenize to zero tokens", failures)    if la["pct_over_max_seq_length"] > 5:        print("   WARN  more than 5%% would be truncated; raise --max_seq_length "              "to about %d" % la["total_tokens_p99"])    # ---------------- 8. sample rendering --------------------------------    print("\n8. Sample rendering (one per question type)")    seen = set()    for row in splits["train"]:        t = row.get("question_type")        if t in seen:            continue        pc = AD.format_example(row, tok, a.completion_style)        if not pc["completion"]:            continue        seen.add(t)        print("   --- %s [%s] ---" % (t, AD.infer_language(row)))        print("   prompt tail : ...%s"              % pc["prompt"][-160:].replace("\n", " | "))        print("   completion  : %r" % pc["completion"][:80])        if len(seen) >= 6:            break    # ---------------- 9. plotting path -----------------------------------    if a.plot:        print("\n9. Plot")        import matplotlib        matplotlib.use("Agg")        import matplotlib.pyplot as plt        plt.figure(figsize=(8, 5))        plt.hist(la["all_token_lengths"], bins=50, color="tab:olive")        plt.axvline(a.max_seq_length, color="tab:red", ls="--",                    label="max_seq_length=%d" % a.max_seq_length)        plt.axvline(la["total_tokens_p99"], color="tab:orange", ls=":",                    label="p99=%d" % la["total_tokens_p99"])        plt.xlabel("Tokens (prompt + completion)"); plt.ylabel("Rows")        plt.title("Token length distribution"); plt.legend(); plt.grid(alpha=0.3)        plt.tight_layout()        plt.savefig(a.plot, dpi=140)        plt.close()        print("   wrote %s" % a.plot)    print("\n%s" % ("PASS --- data layer is sound" if not failures                    else "FAIL --- %d problem(s):" % len(failures)))    for f in failures:        print("  - %s" % f)    sys.exit(0 if not failures else 1)if __name__ == "__main__":    main()

In [ ]:
!python test_ayur_data.py --data_dir $DATA_DIR --max_seq_length 1024 || echo 'Fix this before running the trainer.'

## 6 --- Point at your dataAttach your dataset under **Add-ons -> Datasets**, then set `DATA_DIR`.Two layouts work:- **QA splits** --- a folder with `train.json`, `validation.json`, `test.json`- **Raw treatise** --- a folder with `FINAL_*.jsonl`; set `TRAIN_FILE` to one of  them and leave `validation.json` absent. The trainer carves an eval split by  `chunk_id`, so no verse lands in both halves.

In [ ]:
import os, globDATA_DIR   = "/kaggle/input/ayur-splits"       # <-- your dataset folderTRAIN_FILE = "train.json"                       # or e.g. FINAL_Charaka_Samhita.jsonlEVAL_FILE  = "validation.json"                  # may be absent for raw chunksBASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"os.environ["DATA_DIR"] = DATA_DIRprint("contents of", DATA_DIR)for p in sorted(glob.glob(os.path.join(DATA_DIR, "*")))[:20]:    print("  %-60s %8.1f MB" % (os.path.basename(p), os.path.getsize(p) / 1e6))

## 7 --- Smoke run200 samples, one epoch. Proves the model loads, the masking check passes, a stepruns and the plots render. Do this before the real run every time you change themodel or the data.

In [ ]:
!python finetune_ayur.py \    --model_name_or_path $BASE_MODEL \    --data_dir $DATA_DIR \    --train_file $TRAIN_FILE \    --eval_file $EVAL_FILE \    --max_train_samples 200 \    --max_eval_samples 50 \    --max_seq_length 512 \    --output_dir /kaggle/working/ayur_smoke \    --per_device_train_batch_size 2 \    --gradient_accumulation_steps 4 \    --learning_rate 2e-4 \    --num_train_epochs 1 \    --logging_steps 2 \    --save_steps 50 \    --lora_r 16 --lora_alpha 32

In [ ]:
# the shell cell above needs these exportedimport osos.environ["BASE_MODEL"] = BASE_MODELos.environ["TRAIN_FILE"] = TRAIN_FILEos.environ["EVAL_FILE"]  = EVAL_FILEprint("exported:", BASE_MODEL, "|", DATA_DIR, "|", TRAIN_FILE)

## 8 --- Full run`--auto_batch_size` sizes the batch from free VRAM instead of you guessing. Ifyou hit OOM, raise `--gradient_accumulation_steps` rather than lowering the batch--- the effective batch stays the same and the loss curve does not change shape.Kaggle sessions expire. The trainer auto-resumes from the last checkpoint in`--output_dir`, so re-running this cell after a timeout picks up where itstopped.

In [ ]:
!python finetune_ayur.py \    --model_name_or_path $BASE_MODEL \    --data_dir $DATA_DIR \    --train_file $TRAIN_FILE \    --eval_file $EVAL_FILE \    --max_seq_length 1024 \    --output_dir /kaggle/working/ayur_qlora \    --auto_batch_size \    --gradient_accumulation_steps 8 \    --learning_rate 2e-4 \    --lr_scheduler_type cosine \    --warmup_ratio 0.03 \    --num_train_epochs 2 \    --logging_steps 10 \    --save_steps 200 \    --save_total_limit 3 \    --lora_r 32 --lora_alpha 64

## 9 --- Dashboards

In [ ]:
from IPython.display import Image, displayimport os, jsonOUT = "/kaggle/working/ayur_qlora"plots = os.path.join(OUT, "training_plots")for name in ["dashboard.png", "loss_curve.png", "gpu_memory_spikes.png"]:    p = os.path.join(plots, name)    if os.path.exists(p):        print("---", name, "---")        display(Image(filename=p))for f in ["preflight_report.json", "final_metrics.json"]:    p = os.path.join(OUT, f)    if os.path.exists(p):        print("\n===", f, "===")        d = json.load(open(p))        print(json.dumps(d, indent=2, ensure_ascii=False)[:2000])

## 10 --- Save the adapterThe LoRA adapter is a few hundred MB, not the whole model. Keep it as an adapterunless you specifically need a merged checkpoint --- merging costs full model sizeon disk and you lose the ability to swap adapters.

In [ ]:
import shutil, osOUT = "/kaggle/working/ayur_qlora"shutil.make_archive("/kaggle/working/ayur_lora_adapter", "zip", OUT)print("zipped ->", "/kaggle/working/ayur_lora_adapter.zip",      "%.1f MB" % (os.path.getsize("/kaggle/working/ayur_lora_adapter.zip") / 1e6))# push to the Hub instead, if you set HF_TOKEN above# from peft import PeftModel# PeftModel.from_pretrained(base, OUT).push_to_hub("your-name/ayurllm-lora")

## 11 --- Quick sanity generationLoad the adapter and ask it something from the corpus. Nonsense here after aloss curve that looked fine usually means the prompt format at inference does notmatch training --- check that you are using the same chat template and the samesystem prompt.

In [ ]:
import torchfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfigfrom peft import PeftModelimport ayur_data as ADtok = AutoTokenizer.from_pretrained(BASE_MODEL)bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",                         bnb_4bit_compute_dtype=torch.float16)base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,                                            device_map="auto")model = PeftModel.from_pretrained(base, "/kaggle/working/ayur_qlora").eval()user = ("Translate this verse from Charaka Samhita, Chikitsasthana, Chapter 1 "        "into English.\n\nअथातोऽभयामलकीयं रसायनपादं व्याख्यास्यामः॥ १ ॥")msgs = [{"role": "system", "content": AD.SYSTEM["en"]},        {"role": "user", "content": user}]ids = tok.apply_chat_template(msgs, return_tensors="pt",                              add_generation_prompt=True).to(model.device)with torch.no_grad():    out = model.generate(ids, max_new_tokens=160, do_sample=False)print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))